# AgriEasy Saathi — free QLoRA fine-tune (Colab / Kaggle)

**Setup:** Colab → Runtime → Change runtime type → **T4 GPU** (free). Kaggle → Settings → Accelerator → **GPU T4 x2**, Internet → ON.

Then Run all. 1.5B ≈ 1–2 h, 3B ≈ 3–5 h on free tier. Checkpoints save to disk, so a disconnect loses little.

Model sizes: `qwen1.5b` (phone/PWA target) · `qwen3b` (best quality/speed for free tier) · `qwen7b` / `llama8b` (max free ceiling — Kaggle T4x2 recommended)

In [ ]:
# 1) Install trainer (fast: no-deps trick avoids slow pip resolution on py3.13)
%pip install -q --no-deps unsloth
%pip install -q --no-deps trl peft accelerate
%pip install -q bitsandbytes datasets
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU!')

In [ ]:
import os, pathlib, subprocess
REPO_URL = "https://github.com/CHANGE-ME/agrieasy-ai"
HF_TOKEN = ""   # set to mirror live logs to run/live.log
BEACON_REPO = "Rishabhthe1/agrieasy-ai-source"
def beacon(msg):
    print("[beacon]", msg)
    if HF_TOKEN:
        try:
            from huggingface_hub import HfApi
            HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=msg.encode(), path_in_repo="run/live.log", repo_id=BEACON_REPO, repo_type="dataset")
            print("[beacon] mirrored")
        except Exception as e: print("[beacon] err:", e)
beacon("session started")
r = subprocess.run(["git","clone","-q",REPO_URL])
if r.returncode != 0: raise SystemExit("clone failed")
cand = [d for d in pathlib.Path(".").iterdir() if d.is_dir() and (d/"finetune"/"train_qlora.py").exists()]
if not cand: raise SystemExit("repo layout unexpected")
os.chdir(cand[0])
print("workdir:", os.getcwd())
beacon("cloned ok, workdir=" + os.getcwd())

In [ ]:
r = subprocess.run("python finetune/build_dataset.py --kb_dir knowledge --seed finetune/seed_qa.jsonl --out finetune/data", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0: raise SystemExit("dataset build failed")
beacon("dataset built: " + r.stdout.strip().splitlines()[-1])

In [ ]:
# 4) Train — streams output AND mirrors progress to Hugging Face (run/live.log) every 30s
#    so logs can be watched live from outside Kaggle.
import subprocess, threading, time, os
MODEL = "qwen1.5b"
HF_TOKEN = ""        # optional: hf_xxx write token
HF_REPO  = ""        # optional: your-username/agrieasy-ai-v1
BEACON_REPO = "Rishabhthe1/agrieasy-ai-source"   # dataset repo that mirrors live logs
cmd = f"python finetune/train_qlora.py --model {MODEL} --data_dir finetune/data --out finetune/agrieasy-ai-v1"
if HF_TOKEN and HF_REPO:
    os.environ["HF_TOKEN"] = HF_TOKEN
    cmd += f" --push --hf-repo {HF_REPO}"
buf = []
def beacon():
    if not HF_TOKEN: return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    while proc.poll() is None:
        time.sleep(30)
        try: api.upload_file(path_or_fileobj=("\n".join(buf[-300:])).encode(), path_in_repo="run/live.log", repo_id=BEACON_REPO, repo_type="dataset")
        except Exception as e: print("beacon:", e)
print('>>>', cmd, '\n')
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=beacon, daemon=True).start()
for line in proc.stdout:
    print(line, end=""); buf.append(line.rstrip())
proc.wait()
print("exit code:", proc.returncode)


In [ ]:
# 5) Talk to your fine-tuned model
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained("finetune/agrieasy-ai-v1", max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
q = "Meri gehun ki 2 ekad zameen hai, poora plan banayein beej se bikri tak"
ids = tok.apply_chat_template([{"role":"user","content":q}], return_tensors="pt", add_generation_prompt=True).to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=400, temperature=0.5)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## Next steps
- **Publish:** ran with HF_TOKEN? Your model is at `huggingface.co/<you>/agrieasy-ai-v1`.
- **Into the PWA:** see `finetune/export_webllm.md` (1.5B → in-browser via WebLLM) — or keep the stock 1.5B and let the system prompt + knowledge base do the work.
- **Errors?** Read the traceback, fix the script, re-run.